## Check domain shift for quality grading model

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from nako_eye.utils import plot
from nako_eye.utils.helpers import muks_1d

device = 'cuda:0'
pd.options.display.float_format = "{:,.2f}".format
plot.set_rc_params(kind='paper', notebook_dpi=120)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load embeddings from quality grading model

In [ ]:
parquet_path = Path('data/quality_embeddings/quality_embeddings.parquet')
df = pd.read_parquet(parquet_path)

print(f"Shape: {df.shape}")
print(f"Models: {df['model'].unique()}")
print(f"Datasets: {df['dataset'].unique()}")
df.head(3)

### Plot embeddings for one model of the ensemble

In [ ]:
# Select one model and extract embeddings per dataset
model_name = df['model'].unique()[7]  # change index to pick a different model
df_model = df[df['model'] == model_name].copy()

X = np.stack(df_model['X'].values)
X_2d = np.stack(df_model['X_2d'].values)
datasets = df_model['dataset'].values
labels = df_model['label'].values
confidences = df_model['confidence'].values

print(f"Model: {model_name}")
print(f"Embedding shape: {X.shape}")
print(f"Datasets: {np.unique(datasets, return_counts=True)}")


dataset_map = {'nako': 0, 'deepdrid': 1}
y = np.column_stack([
    np.vectorize(dataset_map.get)(datasets),
    confidences,
])

In [ ]:
# Plot embeddings
n_subplots = (1, 2)
fig_width = 'full'
fig_height_ratio = 0.5
imbalanced = [False, False]
categorical = [True, False]
feature_label = ['Dataset', 'Score']
mappings = [{0:'NAKO', 1: 'DeepDRiD'}, {}]

fig, ax = plot.plot_embeddings(X_2d, y, mappings, None, n_subplots, fig_width, fig_height_ratio, feature_label, imbalanced, categorical, return_ax=True, suptitle=model_name)

### Plot the distribution of quality scores

In [ ]:
# Distribution of quality scores
palette = dict(zip(['deepdrid', 'nako'], plot.get_palette(library='glasbey', n_colors=2)))

fig, ax = plt.subplots()
plot.set_figsize(fig, 'col', height_ratio=0.8)

sns.kdeplot(data=df, x='confidence', hue='dataset', fill=True, common_norm=False, palette=palette, ax=ax)
ax.get_legend().set_title(None)
plot.detach_axes(ax)
plot.set_labs(ax, xlabs='Confidence', ylabs='Density')
plt.tight_layout()

### Domain classifier
Classifier on the high dimensional embeddings from the quality filters. High performance = Strong shift.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

domain_proba = np.full(len(df), np.nan)
results = []
for model_name_i in df['model'].unique():
    idx = df.index[df['model'] == model_name_i]
    X_i = np.stack(df.loc[idx, 'X'].values)
    y_i = df.loc[idx, 'dataset'].map(dataset_map).values

    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    proba = cross_val_predict(clf, X_i, y_i, cv=cv, method='predict_proba')[:, 1]
    domain_proba[idx] = proba

    auroc = roc_auc_score(y_i, proba)
    results.append({'model': model_name_i, 'auroc': auroc})

df['domain_proba'] = domain_proba  # out-of-fold P(deepdrid | embedding)
domain_shift_df = pd.DataFrame(results).sort_values('auroc', ascending=False)
domain_shift_df

In [ ]:
# AUROC of the domain classifier per model (0.5 = indistinguishable, 1.0 = fully separable)
model_labels = domain_shift_df['model'].str.replace('model_', '').str.replace('_', ': ')

fig, ax = plt.subplots()
plot.set_figsize(fig, 'col', height_ratio=0.8)

ax.barh(model_labels, domain_shift_df['auroc'], color=palette['deepdrid'])
ax.invert_yaxis()  # highest AUROC on top
ax.axvline(0.5, color='k', linestyle='--', lw=1)
ax.set_xlim(0.4, 1)
ax.set_xlabel('AUROC')
ax.set_title('Domain classifier on embedding space (DeepDRiD vs NAKO)')
plot.detach_axes(ax)
plt.tight_layout()

Classifier on the final score given by the model. High performance = strong shift.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

domain_prob = np.full(len(df), np.nan)
results = []

X_conf = df['confidence'].to_numpy()
y_dataset = df['dataset'].map(dataset_map).to_numpy()

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
domain_prob = cross_val_predict(clf, X_conf[:, None], y_dataset, cv=cv, method='predict_proba')

auroc = roc_auc_score(y_dataset, domain_prob[:, 1])
results.append({'model': model_name_i, 'auroc': auroc})

df['domain_prob'] = domain_prob[:, 1]  # out-of-fold P(deepdrid | embedding)
print(f'AUROC = {auroc:.3f}')

### MUKS: Mass-Univariate Kolmogorov-Smirnov Tests on Task Predictions

In [ ]:
sample_sizes = [10, 30, 50, 100, 200, 500]  # capped by your pool of 3000
scores_deepdrid = df[df['dataset'] == 'deepdrid']['confidence'].to_numpy()
scores_nako = df[df['dataset'] == 'nako']['confidence'].to_numpy()

results = []
for m in sample_sizes:
    power, type1_err = muks_1d(scores_deepdrid, scores_nako, sample_size=m, num_repetitions=500)
    results.append({'m': m, 'power': power, 'type1_err': type1_err})

muks_df = pd.DataFrame(results)
muks_df

In [ ]:
# Power and Type I error as a function of sample size m
fig, ax = plt.subplots()
plot.set_figsize(fig, 'col', height_ratio=0.8)

ax.plot(muks_df['m'], muks_df['power'], marker='o', label='Power')
ax.plot(muks_df['m'], muks_df['type1_err'], marker='o', label='Type I error')
ax.axhline(0.05, color='k', linestyle='--', lw=1, label='alpha = 0.05')

ax.set_xscale('log')
ax.set_xticks(muks_df['m'])
ax.set_xticklabels(muks_df['m'])
ax.set_ylim(-0.05, 1.05)
plot.detach_axes(ax)
plot.set_labs(ax, xlabs='Sample size (m)', ylabs='Rejection rate')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()